## Imports

Nesta etapa instalamos/carregamos as ferramentas para ler o corpus, processar português e treinar Word2Vec. A instalação do modelo linguístico do spaCy é necessária apenas uma vez por ambiente Python.


In [1]:
# Instala spaCy no ambiente associado a este notebook (uma vez por ambiente).
%pip install spacy

# Descarrega o modelo de português usando o mesmo interpretador do kernel.
import sys
!{sys.executable} -m spacy download pt_core_news_sm



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 44.7 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')


In [2]:
# Carrega o corpus de sinopses de filmes a partir do Hugging Face Datasets.
from datasets import load_dataset

# Word2Vec é a implementação usada para treinar os embeddings.
from gensim.models import Word2Vec
import gensim

# spaCy fará a tokenização, lematização e identificação de stopwords.
import spacy

# O dataset tem divisões; aqui usamos a divisão de treino.
dataset = load_dataset("tclopess/sinopsys_movies_portuguese")

# Carrega o modelo linguístico português previamente instalado.
nlp = spacy.load("pt_core_news_sm")


## Get Data

Selecionamos as sinopses que servirão de corpus. Cada sinopse será tratada como uma sequência independente de tokens; o Word2Vec aprende com as vizinhanças dentro de cada sequência.


In [3]:
# Converte a divisão de treino para uma tabela pandas.
# Esta célula usa todas as linhas disponíveis; não está a limitar a 3000 exemplos.
df_sinop = dataset["train"].to_pandas()

# Inspeciona as primeiras linhas e confirma o nome/forma das colunas.
df_sinop.head()


,titulo,sinopse,generos,is_valid
0,We Were Soldiers,A história da primeira grande batalha da fase ...,"['Ação', 'História', 'Guerra']",False
1,4Got10,"Um negócio de drogas dá errado, deixando corpo...","['Ação', 'Crime', 'Thriller']",False
2,Pontypool,Quando o disc jockey Grant Mazzy se reporta à ...,"['Horror', 'Mistério', 'Ficção Científica']",False
3,Ticker,Depois que o parceiro de um detetive de São Fr...,"['Ação', 'Crime', 'Thriller']",False
4,Real Genius,Um adolescente prodígio tenso entra em uma fac...,"['Comédia', 'Romance', 'Ficção Científica']",True


In [4]:
# Pré-processamento configurável: mantém stopwords por omissão para preservar contexto.
# Experimente True e False e compare os embeddings; remover "não", por exemplo,
# pode inverter ou apagar informação importante numa frase.
REMOVE_STOPWORDS = False

def clean_text(text):
    # O spaCy separa o texto em tokens e acrescenta informação linguística.
    doc = nlp(text)

    # Converte para minúsculas e lemas para juntar formas como "gatos" e "gato".
    # Pontuação é removida; a opção abaixo controla separadamente as stopwords.
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct and (not REMOVE_STOPWORDS or not token.is_stop)
    ]

    # Word2Vec recebe uma sequência de tokens por sinopse.
    return tokens


In [5]:
# Aplica o pré-processamento a cada sinopse.
# O resultado é uma série em que cada elemento é uma lista de tokens.
data = df_sinop["sinopse"].apply(clean_text)


In [6]:
# Inspeciona os tokens da primeira sinopse para verificar o pré-processamento.
data.iloc[0]


['o',
 'história',
 'de o',
 'primeiro',
 'grande',
 'batalha',
 'de o',
 'fase',
 'americano',
 'de o',
 'guerra',
 'de o',
 'vietnã',
 'e',
 'o',
 'soldado',
 'de',
 'ambos',
 'o',
 'lado',
 'que',
 'ela',
 'travar']

## Create Word2Vec Model

Cada item de `data` é uma sequência de palavras. O Gensim cria janelas de contexto dentro dessas sequências e aprende vetores que ajudam a prever palavras vizinhas. Por omissão, `sg=0`, portanto o primeiro modelo é CBOW.


In [7]:
# Treina o primeiro modelo: CBOW é o padrão do Gensim (sg=0).
model1 = Word2Vec(
    sentences=data,    # sequências de tokens; cada sinopse é uma sequência
    vector_size=100,   # dimensão do embedding de cada palavra
    window=4,          # até 4 tokens de contexto de cada lado da palavra-alvo
    min_count=2,       # ignora palavras que aparecem menos de 2 vezes
    workers=4          # número de processos usados no treino
)
# O treino acontece durante a criação do objeto Word2Vec.


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [8]:
# O vocabulário contém as palavras que sobreviveram ao filtro min_count.
words = list(model1.wv.index_to_key)
print("Dimensão do vocabulário:", len(words))
print("Primeiras 10 palavras:", words[:10])


Dimensão do vocabulário: 18401
Primeiras 10 palavras: ['um', 'de', 'o', 'e', 'de o', 'seu', 'que', 'para', 'ser', 'em']


In [9]:
# most_similar procura palavras com vetores próximos segundo similaridade de cosseno.
# A palavra consultada tem de existir no vocabulário (model1.wv).
#https://radimrehurek.com/gensim/models/keyedvectors.html
similar_words = model1.wv.most_similar("guerra", topn=10)
print("Palavras mais semelhantes a 'guerra':")
for word, similarity in similar_words:
    # O score é uma similaridade, não uma probabilidade.
    print(f"{word}: {similarity:.4f}")


Palavras mais semelhantes a 'guerra':
mundial: 0.8580
segunda: 0.8486
batalha: 0.8235
dominação: 0.8146
renome: 0.8068
vietnã: 0.8035
golfo: 0.7759
civil: 0.7734
bósnia: 0.7653
laos: 0.7576


In [10]:
# Repete a consulta para outra palavra-alvo.
similar_words = model1.wv.most_similar("amor", topn=10)
print("Palavras mais semelhantes a 'amor':")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")


Palavras mais semelhantes a 'amor':
verdadeiro: 0.8701
identidade: 0.8641
natureza: 0.8484
pessoal: 0.8271
relacionamento: 0.8180
medo: 0.8146
passado: 0.8125
crença: 0.8122
adulto: 0.8105
dúvida: 0.8096


In [11]:
# Treina outro modelo CBOW, alterando apenas a dimensão do vetor para 300.
# Serve para comparar embeddings de 100 e 300 dimensões.
model2 = Word2Vec(
    sentences=data,
    min_count=2,
    vector_size=300,
    window=4
)


In [12]:
# Consulta o modelo de 300 dimensões para comparar com model1 (100 dimensões).
similar_words = model2.wv.most_similar("guerra", topn=10)
print("model2 — palavras semelhantes a 'guerra':")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")


model2 — palavras semelhantes a 'guerra':
mundial: 0.8758
segunda: 0.8667
dominação: 0.8353
batalha: 0.8274
ca: 0.8223
vietnã: 0.8118
renome: 0.8105
civil: 0.8006
laos: 0.7933
bósnia: 0.7926


In [13]:
# Compara também a vizinhança de "amor" no modelo model2.
similar_words = model2.wv.most_similar("amor", topn=10)
print("model2 — palavras semelhantes a 'amor':")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")


model2 — palavras semelhantes a 'amor':
verdadeiro: 0.8899
identidade: 0.8606
natureza: 0.8440
pessoal: 0.8342
dúvida: 0.8294
relacionamento: 0.8282
medo: 0.8185
passado: 0.8144
crença: 0.8096
significado: 0.8092


In [14]:
# Treina Skip-gram: sg=1 faz o modelo prever palavras de contexto a partir da palavra-alvo.
# Mantemos dimensão, janela e frequência mínima iguais a model2 para comparar a arquitetura.
model3 = Word2Vec(
    sentences=data,
    min_count=2,
    vector_size=300,
    window=4,
    sg=1
)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [15]:
# Consulta model3 (Skip-gram) e compara com os resultados dos modelos CBOW.
similar_words = model3.wv.most_similar("guerra", topn=10)
print("model3 Skip-gram — palavras semelhantes a 'guerra':")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")


model3 Skip-gram — palavras semelhantes a 'guerra':
mundial: 0.7390
segunda: 0.7347
civil: 0.7025
batalha: 0.6950
alemão: 0.6886
nazista: 0.6844
americana: 0.6842
hitler: 0.6832
vietnã: 0.6767
concentração: 0.6609


## Representation and Visualization

Vamos visualizar em duas dimensões alguns vetores de palavras semelhantes a `amor` e `guerra`. PCA comprime os vetores de 100 dimensões para 2 componentes para desenhar; a posição no gráfico é uma aproximação e pode perder relações presentes no espaço original.


In [16]:
# Obtém as 10 palavras mais semelhantes a cada consulta, usando model1.
amor_words = model1.wv.most_similar("amor", topn=10)
guerra_words = model1.wv.most_similar("guerra", topn=10)

# Extrai apenas os rótulos das palavras para o hover/legenda.
amor_labels = [word for word, similarity in amor_words]
guerra_labels = [word for word, similarity in guerra_words]

# Obtém o embedding de cada palavra; cada vetor tem 100 componentes neste modelo.
amor_vectors = [model1.wv[word] for word in amor_labels]
guerra_vectors = [model1.wv[word] for word in guerra_labels]


In [17]:
# Junta os dois grupos antes do PCA para que partilhem os mesmos eixos.
# Ajustar um PCA separado para cada grupo tornaria as posições incomparáveis.
import numpy as np
from sklearn.decomposition import PCA

all_vectors = np.vstack([amor_vectors, guerra_vectors])
all_labels = amor_labels + guerra_labels
groups = ["amor"] * len(amor_labels) + ["guerra"] * len(guerra_labels)

# Reduz os vetores de 100 dimensões para duas coordenadas apenas para visualização.
pca = PCA(n_components=2)
reduced = pca.fit_transform(all_vectors)


In [18]:
# Organiza coordenadas e rótulos numa tabela para o gráfico.
import pandas as pd

plot_df = pd.DataFrame({
    "PC1": reduced[:, 0],
    "PC2": reduced[:, 1],
    "palavra": all_labels,
    "grupo": groups,
})
plot_df.head()


,PC1,PC2,palavra,grupo
0,2.008797,-2.904613,verdadeiro,amor
1,1.692982,-1.174038,identidade,amor
2,0.850649,0.289817,natureza,amor
3,0.996823,-0.733677,pessoal,amor
4,2.397012,-2.695082,relacionamento,amor


In [19]:
# Instala o Plotly e o suporte de formatos do notebook no kernel atual.
# %pip instala no ambiente Python associado a este notebook.
%pip install plotly "nbformat>=4.2.0"



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
# Plotly cria um gráfico interativo; passe o cursor sobre qualquer ponto
# para ver sempre a palavra, o grupo e as coordenadas reduzidas.
import plotly.express as px

fig = px.scatter(
    plot_df,
    x="PC1",
    y="PC2",
    color="grupo",
    custom_data=["palavra", "grupo"],
    title="PCA — palavras relacionadas com 'amor' e 'guerra'",
)

# Define explicitamente o conteúdo do tooltip para todas as séries/pontos.
fig.update_traces(
    marker={"size": 12, "opacity": 0.9},
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Grupo: %{customdata[1]}<br>"
        "PC1: %{x:.3f}<br>"
        "PC2: %{y:.3f}"
        "<extra></extra>"
    ),
)

# Mostra apenas o tooltip do ponto mais próximo do cursor.
fig.update_layout(hovermode="closest")
fig.show()


In [21]:
# Guarda o gráfico interativo num HTML autónomo, que pode ser aberto num browser.
from pathlib import Path

# Se o kernel estiver na raiz do projeto, guarda junto dos materiais da Week 7.
# Se estiver a executar dentro de 07_Week7, usa essa pasta diretamente.
week7_dir = Path.cwd() / "07_Week7"
output_dir = week7_dir if week7_dir.is_dir() else Path.cwd()
output_dir.mkdir(parents=True, exist_ok=True)
html_path = output_dir / "word2vec_pca.html"

# include_plotlyjs=True incorpora Plotly no ficheiro; não depende de internet.
fig.write_html(str(html_path), full_html=True, include_plotlyjs=True)
print(f"Gráfico guardado em: {html_path.resolve()}")


Gráfico guardado em: /Users/zuilpirola/Documents/Documents - Zuil’s MacBook Pro/01_projects/fund_NLP/07_Week7/word2vec_pca.html
